# ARC-AGI End-to-End (Kaggle, PyTorch, <= 50M Params)

This notebook builds a complete ARC-AGI pipeline under strict constraints:
- Train from scratch
- Use ARC training set only for training
- Evaluate on ARC evaluation set using exact match only
- Two predictions per test case, task solved if either matches exactly
- Transformer encoder-decoder with required dimensions
- Total parameters <= 50M

References used for design decisions:
1. https://arxiv.org/abs/2510.04871
2. https://arxiv.org/html/2512.06104
3. https://arxiv.org/html/2506.21734v1
4. https://mvakde.github.io/blog/why-all-ARC-solvers-fail-today/
5. https://mvakde.github.io/blog/44-on-arc-1/

In [ ]:
import subprocess, sys, os, glob, math, random, time, copy, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print("=== Environment ===")
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Detect environment — works on Kaggle and Colab
if os.path.exists('/kaggle/working'):
    CHECKPOINT_DIR = '/kaggle/working/ARC_checkpoints'
    ARC_ROOT       = '/kaggle/working/ARC-AGI'
else:
    # For Colab with Drive persistence (uncomment two lines below):
    # from google.colab import drive; drive.mount('/content/drive')
    # CHECKPOINT_DIR = '/content/drive/MyDrive/ARC_checkpoints'
    CHECKPOINT_DIR = '/content/ARC_checkpoints'
    ARC_ROOT       = '/content/ARC-AGI'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"\nCheckpoints : {CHECKPOINT_DIR}")

In [ ]:
# Cell 2 — Download ARC-AGI-1 dataset from GitHub
TRAIN_DIR = f'{ARC_ROOT}/data/training'
EVAL_DIR  = f'{ARC_ROOT}/data/evaluation'

if not os.path.exists(ARC_ROOT):
    os.system(f'git clone --quiet https://github.com/fchollet/ARC-AGI.git {ARC_ROOT}')

print(f"Training tasks  : {len(os.listdir(TRAIN_DIR))}")
print(f"Evaluation tasks: {len(os.listdir(EVAL_DIR))}")

# Quick sanity check
_sample_file = sorted(os.listdir(TRAIN_DIR))[0]
with open(os.path.join(TRAIN_DIR, _sample_file)) as f:
    _s = json.load(f)
_d = _s['train'][0]
print(f"\nSample: {_sample_file}  "
      f"| {len(_s['train'])} demos "
      f"| inp {len(_d['input'])}×{len(_d['input'][0])} "
      f"| out {len(_d['output'])}×{len(_d['output'][0])}")

## 1) Configuration

In [ ]:
@dataclass
class Config:
    # Dataset paths are set from the ARC-AGI GitHub clone in the download cell.
    train_dir: str = ''
    eval_dir: str = ''

    seed: int = 42

    # Tokenization / sequence
    max_seq_len: int = 512
    max_pairs: int = 10

    # Model (requested)
    d_model: int = 512
    num_heads: int = 8
    encoder_layers: int = 8
    decoder_layers: int = 8
    feedforward_dim: int = 2048
    dropout: float = 0.1

    # Keep params <=50M by sharing layer weights across depth
    share_encoder_layers: bool = True
    share_decoder_layers: bool = True

    # Training
    batch_size: int = 8
    grad_accum_steps: int = 2
    num_epochs: int = 20
    lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    max_steps: int = 20000
    val_every_steps: int = 500
    patience: int = 6

    # Augmentation
    use_augmentation: bool = True
    color_perm_prob: float = 0.20

    # Decoding
    max_decode_tokens: int = 256
    temperature: float = 0.9
    top_k: int = 20

    # Optional test-time adaptation
    enable_ttt: bool = False
    ttt_steps: int = 10
    ttt_lr: float = 1e-5

    # Ablations
    run_ablations: bool = True

cfg = Config()
cfg.train_dir = TRAIN_DIR
cfg.eval_dir = EVAL_DIR
print(cfg)

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2) Data Loading (ARC JSON)

In [ ]:
def load_arc_tasks(folder: str):
    folder_path = Path(folder)
    assert folder_path.exists(), f'Missing folder: {folder}'
    tasks = []
    for fp in sorted(folder_path.glob('*.json')):
        with open(fp, 'r') as f:
            obj = json.load(f)
        task = {
            'task_id': fp.stem,
            'train': obj.get('train', []),
            'test': obj.get('test', []),
        }
        tasks.append(task)
    return tasks

train_tasks_all = load_arc_tasks(TRAIN_DIR)
eval_tasks_all = load_arc_tasks(EVAL_DIR)
print('Train tasks:', len(train_tasks_all))
print('Eval tasks:', len(eval_tasks_all))
assert len(train_tasks_all) == 400, 'Expected 400 ARC training tasks'
assert len(eval_tasks_all) == 400, 'Expected 400 ARC evaluation tasks'

In [ ]:
def train_val_split(tasks, val_ratio=0.1, seed=42):
    ids = list(range(len(tasks)))
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_val = int(len(ids) * val_ratio)
    val_idx = set(ids[:n_val])
    train_tasks = [tasks[i] for i in range(len(tasks)) if i not in val_idx]
    val_tasks = [tasks[i] for i in range(len(tasks)) if i in val_idx]
    return train_tasks, val_tasks

# Validation is split only from the 400 training tasks; evaluation tasks remain untouched until final testing.
train_tasks, val_tasks = train_val_split(train_tasks_all, val_ratio=0.1, seed=cfg.seed)
print('Train tasks for fitting:', len(train_tasks))
print('Validation tasks from training split:', len(val_tasks))
print('Evaluation tasks reserved for final test:', len(eval_tasks_all))

## 3) Tokenizer + Preprocessing

In [ ]:
class Tokenizer:
    """ARC tokenizer with structural tokens and feature channels.

    Token channels per position:
    - token_id (color or special token)
    - row_id (0..30)
    - col_id (0..30)
    - pair_id (0..max_pairs)
    - io_id (0 control, 1 input, 2 output, 3 test)
    """

    def __init__(self, max_pairs=10):
        self.max_pairs = max_pairs

        specials = [
            '<pad>', '<start>', '<sep>', '<input>', '<output>', '<test>',
            '<pair_start>', '<end>'
        ]
        colors = [f'C{i}' for i in range(10)]
        heights = [f'H{i}' for i in range(1, 31)]
        widths = [f'W{i}' for i in range(1, 31)]

        vocab = specials + colors + heights + widths
        self.stoi = {t: i for i, t in enumerate(vocab)}
        self.itos = {i: t for t, i in self.stoi.items()}

        self.pad_id = self.stoi['<pad>']
        self.start_id = self.stoi['<start>']
        self.sep_id = self.stoi['<sep>']
        self.input_id = self.stoi['<input>']
        self.output_id = self.stoi['<output>']
        self.test_id = self.stoi['<test>']
        self.pair_start_id = self.stoi['<pair_start>']
        self.end_id = self.stoi['<end>']

        self.vocab_size = len(vocab)

    def color_token(self, c):
        return self.stoi[f'C{int(c)}']

    def height_token(self, h):
        h = int(np.clip(h, 1, 30))
        return self.stoi[f'H{h}']

    def width_token(self, w):
        w = int(np.clip(w, 1, 30))
        return self.stoi[f'W{w}']

    def _append_control(self, seq, token_id, pair_id=0, io_id=0):
        seq['token'].append(token_id)
        seq['row'].append(0)
        seq['col'].append(0)
        seq['pair'].append(pair_id)
        seq['io'].append(io_id)

    def _append_grid(self, seq, grid, pair_id, io_id):
        h = len(grid)
        w = len(grid[0])
        self._append_control(seq, self.height_token(h), pair_id=pair_id, io_id=io_id)
        self._append_control(seq, self.width_token(w), pair_id=pair_id, io_id=io_id)

        for r in range(h):
            for c in range(w):
                seq['token'].append(self.color_token(grid[r][c]))
                seq['row'].append(r + 1)
                seq['col'].append(c + 1)
                seq['pair'].append(pair_id)
                seq['io'].append(io_id)

    def encode_source(self, train_pairs, test_input, max_len=512):
        seq = {k: [] for k in ['token', 'row', 'col', 'pair', 'io']}
        self._append_control(seq, self.start_id)

        for i, p in enumerate(train_pairs, start=1):
            pid = min(i, self.max_pairs)
            self._append_control(seq, self.pair_start_id, pair_id=pid, io_id=0)

            self._append_control(seq, self.input_id, pair_id=pid, io_id=1)
            self._append_grid(seq, p['input'], pair_id=pid, io_id=1)

            self._append_control(seq, self.sep_id, pair_id=pid, io_id=0)

            self._append_control(seq, self.output_id, pair_id=pid, io_id=2)
            self._append_grid(seq, p['output'], pair_id=pid, io_id=2)

            self._append_control(seq, self.sep_id, pair_id=pid, io_id=0)

        test_pid = min(len(train_pairs) + 1, self.max_pairs)
        self._append_control(seq, self.pair_start_id, pair_id=test_pid, io_id=0)
        self._append_control(seq, self.test_id, pair_id=test_pid, io_id=3)
        self._append_grid(seq, test_input, pair_id=test_pid, io_id=3)
        self._append_control(seq, self.end_id, pair_id=test_pid, io_id=0)

        if len(seq['token']) > max_len:
            return None

        return seq

    def encode_target(self, output_grid, max_len=512):
        seq = {k: [] for k in ['token', 'row', 'col', 'pair', 'io']}
        self._append_control(seq, self.start_id, pair_id=0, io_id=2)

        h = len(output_grid)
        w = len(output_grid[0])
        self._append_control(seq, self.height_token(h), pair_id=0, io_id=2)
        self._append_control(seq, self.width_token(w), pair_id=0, io_id=2)

        for r in range(h):
            for c in range(w):
                seq['token'].append(self.color_token(output_grid[r][c]))
                seq['row'].append(r + 1)
                seq['col'].append(c + 1)
                seq['pair'].append(0)
                seq['io'].append(2)

        self._append_control(seq, self.end_id, pair_id=0, io_id=2)

        if len(seq['token']) > max_len:
            return None

        return seq

    def decode_output_tokens(self, token_ids):
        # Expect: <start> Hx Wy C... <end>
        toks = [self.itos.get(int(t), '<unk>') for t in token_ids]
        if len(toks) < 5 or toks[0] != '<start>':
            return None

        # Find first H and W after start
        if not toks[1].startswith('H') or not toks[2].startswith('W'):
            return None

        try:
            h = int(toks[1][1:])
            w = int(toks[2][1:])
        except ValueError:
            return None

        needed = h * w
        colors = []
        for tok in toks[3:]:
            if tok == '<end>':
                break
            if tok.startswith('C'):
                try:
                    colors.append(int(tok[1:]))
                except ValueError:
                    pass

        if len(colors) < needed:
            return None

        colors = colors[:needed]
        grid = []
        idx = 0
        for _ in range(h):
            row = colors[idx:idx + w]
            grid.append(row)
            idx += w
        return grid

tokenizer = Tokenizer(max_pairs=cfg.max_pairs)
print('Vocab size:', tokenizer.vocab_size)

## 4) Data Augmentation

In [ ]:
def rot90(grid):
    arr = np.array(grid, dtype=np.int64)
    return np.rot90(arr, k=1).tolist()

def rot180(grid):
    arr = np.array(grid, dtype=np.int64)
    return np.rot90(arr, k=2).tolist()

def rot270(grid):
    arr = np.array(grid, dtype=np.int64)
    return np.rot90(arr, k=3).tolist()

def flip_h(grid):
    arr = np.array(grid, dtype=np.int64)
    return np.fliplr(arr).tolist()

def flip_v(grid):
    arr = np.array(grid, dtype=np.int64)
    return np.flipud(arr).tolist()

def apply_transform(grid, tname):
    if tname == 'id':
        return grid
    if tname == 'rot90':
        return rot90(grid)
    if tname == 'rot180':
        return rot180(grid)
    if tname == 'rot270':
        return rot270(grid)
    if tname == 'flip_h':
        return flip_h(grid)
    if tname == 'flip_v':
        return flip_v(grid)
    raise ValueError(tname)

def safe_color_permute(grids, prob=0.2):
    # Keep 0 fixed, permute non-zero colors only with given probability.
    if random.random() > prob:
        return grids

    used = set()
    for g in grids:
        for row in g:
            used.update(row)

    nz = sorted([c for c in used if c != 0])
    if len(nz) <= 1:
        return grids

    perm = nz[:]
    random.shuffle(perm)
    mapping = {0: 0}
    for a, b in zip(nz, perm):
        mapping[a] = b

    out = []
    for g in grids:
        gg = [[mapping.get(v, v) for v in row] for row in g]
        out.append(gg)
    return out

AUGS = ['id', 'rot90', 'rot180', 'rot270', 'flip_h', 'flip_v']

## 5) Dataset Class

In [ ]:
class ARCDataset(Dataset):
    """Expands ARC tasks into (source, target) examples per test case."""

    def __init__(self, tasks, tokenizer, max_seq_len=512,
                 training=True, use_augmentation=True, color_perm_prob=0.2):
        self.tasks = tasks
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.training = training
        self.use_augmentation = use_augmentation
        self.color_perm_prob = color_perm_prob

        self.samples = []
        for t in tasks:
            for test_idx, test_pair in enumerate(t['test']):
                sample = {
                    'task_id': t['task_id'],
                    'train_pairs': t['train'],
                    'test_input': test_pair['input'],
                    'test_output': test_pair.get('output', None),
                    'test_idx': test_idx,
                }
                self.samples.append(sample)

    def __len__(self):
        return len(self.samples)

    def _augment(self, train_pairs, test_input, test_output):
        tname = random.choice(AUGS)

        aug_train = []
        for p in train_pairs:
            aug_train.append({
                'input': apply_transform(p['input'], tname),
                'output': apply_transform(p['output'], tname),
            })

        aug_test_input = apply_transform(test_input, tname)
        aug_test_output = apply_transform(test_output, tname) if test_output is not None else None

        # Optional color permutation over all related grids together.
        all_grids = []
        for p in aug_train:
            all_grids.extend([p['input'], p['output']])
        all_grids.append(aug_test_input)
        if aug_test_output is not None:
            all_grids.append(aug_test_output)

        all_perm = safe_color_permute(all_grids, prob=self.color_perm_prob)

        idx = 0
        new_train = []
        for _ in aug_train:
            new_train.append({
                'input': all_perm[idx],
                'output': all_perm[idx + 1],
            })
            idx += 2
        new_test_input = all_perm[idx]
        idx += 1
        new_test_output = all_perm[idx] if aug_test_output is not None else None

        return new_train, new_test_input, new_test_output

    def __getitem__(self, idx):
        item = self.samples[idx]

        train_pairs = copy.deepcopy(item['train_pairs'])
        test_input = copy.deepcopy(item['test_input'])
        test_output = copy.deepcopy(item['test_output'])

        if self.training and self.use_augmentation:
            train_pairs, test_input, test_output = self._augment(train_pairs, test_input, test_output)

        src = self.tokenizer.encode_source(
            train_pairs=train_pairs,
            test_input=test_input,
            max_len=self.max_seq_len
        )

        # Retry without augmentation if augmented sample exceeds max length.
        if src is None and self.training and self.use_augmentation:
            train_pairs = copy.deepcopy(item['train_pairs'])
            test_input = copy.deepcopy(item['test_input'])
            test_output = copy.deepcopy(item['test_output'])
            src = self.tokenizer.encode_source(train_pairs, test_input, self.max_seq_len)

        if src is None:
            # Return a minimal placeholder; collate will drop invalids.
            return {'valid': False}

        tgt = None
        if test_output is not None:
            tgt = self.tokenizer.encode_target(test_output, max_len=self.max_seq_len)
            if tgt is None:
                return {'valid': False}

        return {
            'valid': True,
            'task_id': item['task_id'],
            'test_idx': item['test_idx'],
            'src': src,
            'tgt': tgt,
            'test_output_grid': test_output,
            'raw_train_pairs': item['train_pairs'],
            'raw_test_input': item['test_input'],
        }

In [ ]:
def pad_1d(seqs, pad_value):
    max_len = max(len(s) for s in seqs)
    out = torch.full((len(seqs), max_len), pad_value, dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out

def collate_fn(batch, pad_id):
    batch = [b for b in batch if b.get('valid', False)]
    if len(batch) == 0:
        return None

    def stack_seq(key, subkey):
        seqs = [b[key][subkey] for b in batch]
        return pad_1d(seqs, 0 if subkey != 'token' else pad_id)

    src = {
        'token': stack_seq('src', 'token'),
        'row': stack_seq('src', 'row'),
        'col': stack_seq('src', 'col'),
        'pair': stack_seq('src', 'pair'),
        'io': stack_seq('src', 'io'),
    }

    has_tgt = batch[0]['tgt'] is not None
    tgt = None
    if has_tgt:
        tgt = {
            'token': stack_seq('tgt', 'token'),
            'row': stack_seq('tgt', 'row'),
            'col': stack_seq('tgt', 'col'),
            'pair': stack_seq('tgt', 'pair'),
            'io': stack_seq('tgt', 'io'),
        }

    return {
        'src': src,
        'tgt': tgt,
        'meta': batch,
    }

In [ ]:
train_ds = ARCDataset(
    tasks=train_tasks,
    tokenizer=tokenizer,
    max_seq_len=cfg.max_seq_len,
    training=True,
    use_augmentation=cfg.use_augmentation,
    color_perm_prob=cfg.color_perm_prob
)

val_ds = ARCDataset(
    tasks=val_tasks,
    tokenizer=tokenizer,
    max_seq_len=cfg.max_seq_len,
    training=False,
    use_augmentation=False,
    color_perm_prob=0.0
)

eval_ds = ARCDataset(
    tasks=eval_tasks_all,
    tokenizer=tokenizer,
    max_seq_len=cfg.max_seq_len,
    training=False,
    use_augmentation=False,
    color_perm_prob=0.0
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_id),
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_id),
)

print('Train task cases expanded to:', len(train_ds))
print('Validation task cases expanded to:', len(val_ds))
print('Evaluation task cases expanded to:', len(eval_ds))

## 6) Transformer Model

In [ ]:
class SharedEncoder(nn.Module):
    def __init__(self, layer, num_layers):
        super().__init__()
        self.layer = layer
        self.num_layers = num_layers

    def forward(self, src, src_key_padding_mask=None):
        x = src
        for _ in range(self.num_layers):
            x = self.layer(x, src_key_padding_mask=src_key_padding_mask)
        return x


class SharedDecoder(nn.Module):
    def __init__(self, layer, num_layers):
        super().__init__()
        self.layer = layer
        self.num_layers = num_layers

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        x = tgt
        for _ in range(self.num_layers):
            x = self.layer(
                x, memory,
                tgt_mask=tgt_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask
            )
        return x


class TransformerModel(nn.Module):
    """Encoder-decoder ARC transformer with composite embeddings."""

    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers,
                 dim_feedforward, dropout, max_seq_len, max_pairs, use_positional=True,
                 share_encoder_layers=True, share_decoder_layers=True):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.use_positional = use_positional

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.row_emb = nn.Embedding(31, d_model)
        self.col_emb = nn.Embedding(31, d_model)
        self.pair_emb = nn.Embedding(max_pairs + 1, d_model)
        self.io_emb = nn.Embedding(5, d_model)

        if use_positional:
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
        else:
            self.pos_emb = None

        self.dropout = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )

        if share_encoder_layers:
            self.encoder = SharedEncoder(enc_layer, num_encoder_layers)
        else:
            self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        if share_decoder_layers:
            self.decoder = SharedDecoder(dec_layer, num_decoder_layers)
        else:
            self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)

        self.norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        # Tie output projection with token embedding for efficiency.
        self.output_proj.weight = self.token_emb.weight

    def embed(self, x):
        tok = self.token_emb(x['token'])
        row = self.row_emb(x['row'])
        col = self.col_emb(x['col'])
        pair = self.pair_emb(torch.clamp(x['pair'], 0, self.pair_emb.num_embeddings - 1))
        io = self.io_emb(torch.clamp(x['io'], 0, self.io_emb.num_embeddings - 1))

        h = tok + row + col + pair + io

        if self.pos_emb is not None:
            bsz, seqlen = x['token'].shape
            pos = torch.arange(seqlen, device=x['token'].device).unsqueeze(0).expand(bsz, seqlen)
            h = h + self.pos_emb(pos)

        return self.dropout(h)

    def _causal_mask(self, L, device):
        m = torch.full((L, L), float('-inf'), device=device)
        return torch.triu(m, diagonal=1)

    def forward(self, src, tgt):
        src_key_padding = src['token'].eq(tokenizer.pad_id)
        tgt_key_padding = tgt['token'].eq(tokenizer.pad_id)

        src_h = self.embed(src)
        tgt_h = self.embed(tgt)

        memory = self.encoder(src_h, src_key_padding_mask=src_key_padding)

        L = tgt_h.shape[1]
        tmask = self._causal_mask(L, tgt_h.device)

        dec = self.decoder(
            tgt_h, memory,
            tgt_mask=tmask,
            tgt_key_padding_mask=tgt_key_padding,
            memory_key_padding_mask=src_key_padding
        )

        dec = self.norm(dec)
        logits = self.output_proj(dec)
        return logits

## 7) Parameter Count (MANDATORY <= 50M)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = TransformerModel(
    vocab_size=tokenizer.vocab_size,
    d_model=cfg.d_model,
    nhead=cfg.num_heads,
    num_encoder_layers=cfg.encoder_layers,
    num_decoder_layers=cfg.decoder_layers,
    dim_feedforward=cfg.feedforward_dim,
    dropout=cfg.dropout,
    max_seq_len=cfg.max_seq_len,
    max_pairs=cfg.max_pairs,
    use_positional=True,
    share_encoder_layers=cfg.share_encoder_layers,
    share_decoder_layers=cfg.share_decoder_layers,
).to(device)

n_params = count_parameters(model)
print(f'Total trainable parameters: {n_params:,}')
print(f'Total trainable parameters (M): {n_params / 1e6:.2f}M')
assert n_params <= 50_000_000, 'Model exceeds 50M parameters'

## 8) Training Utilities (Loss, Optimizer, Scheduler, Checkpointing)

In [ ]:
def move_batch_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if isinstance(v, dict):
            out[k] = {kk: vv.to(device) for kk, vv in v.items()}
        else:
            out[k] = v
    return out

def build_tgt_in_out(tgt):
    # Teacher forcing: input excludes last token, labels exclude first token.
    tgt_in = {k: v[:, :-1] for k, v in tgt.items()}
    tgt_out = tgt['token'][:, 1:]
    return tgt_in, tgt_out

def exact_grid_match(pred_grid, gt_grid):
    if pred_grid is None or gt_grid is None:
        return False
    if len(pred_grid) != len(gt_grid):
        return False
    if len(pred_grid) == 0:
        return False
    if len(pred_grid[0]) != len(gt_grid[0]):
        return False
    return pred_grid == gt_grid

def save_checkpoint(path, model, optimizer, scheduler, scaler, step, best_val):
    payload = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'scaler': scaler.state_dict() if scaler is not None else None,
        'step': step,
        'best_val': best_val,
    }
    torch.save(payload, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None, map_location='cpu'):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt['model'])
    if optimizer is not None and ckpt.get('optimizer') is not None:
        optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler is not None and ckpt.get('scheduler') is not None:
        scheduler.load_state_dict(ckpt['scheduler'])
    if scaler is not None and ckpt.get('scaler') is not None:
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler(enabled=(device.type == 'cuda'))
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)

# Warmup + linear decay scheduler
def lr_lambda(step):
    if step < cfg.warmup_steps:
        return float(step) / float(max(1, cfg.warmup_steps))
    progress = (step - cfg.warmup_steps) / float(max(1, cfg.max_steps - cfg.warmup_steps))
    return max(0.0, 1.0 - progress)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
print('Optimizer and scheduler ready')

## 9) Inference: Greedy + Sampling + Beam (Bonus)

In [ ]:
def make_empty_tgt_features(token_seq):
    # For generated tokens, row/col/pair/io are control defaults.
    # Shape and color structure is learned via token stream itself.
    L = len(token_seq)
    return {
        'token': torch.tensor(token_seq, dtype=torch.long).unsqueeze(0),
        'row': torch.zeros((1, L), dtype=torch.long),
        'col': torch.zeros((1, L), dtype=torch.long),
        'pair': torch.zeros((1, L), dtype=torch.long),
        'io': torch.full((1, L), 2, dtype=torch.long),
    }

@torch.no_grad()
def decode_greedy(model, src, max_new_tokens=256):
    model.eval()
    generated = [tokenizer.start_id]

    src_dev = {k: v.to(device) for k, v in src.items()}

    for _ in range(max_new_tokens):
        tgt = make_empty_tgt_features(generated)
        tgt = {k: v.to(device) for k, v in tgt.items()}

        logits = model(src_dev, tgt)
        next_id = int(torch.argmax(logits[0, -1]).item())
        generated.append(next_id)

        if next_id == tokenizer.end_id:
            break

    return generated

@torch.no_grad()
def decode_sample(model, src, max_new_tokens=256, temperature=0.9, top_k=20):
    model.eval()
    generated = [tokenizer.start_id]
    src_dev = {k: v.to(device) for k, v in src.items()}

    for _ in range(max_new_tokens):
        tgt = make_empty_tgt_features(generated)
        tgt = {k: v.to(device) for k, v in tgt.items()}

        logits = model(src_dev, tgt)[0, -1] / max(temperature, 1e-6)

        if top_k is not None and top_k > 0:
            vals, idx = torch.topk(logits, k=min(top_k, logits.numel()))
            probs = F.softmax(vals, dim=-1)
            pick = idx[torch.multinomial(probs, num_samples=1)].item()
            next_id = int(pick)
        else:
            probs = F.softmax(logits, dim=-1)
            next_id = int(torch.multinomial(probs, num_samples=1).item())

        generated.append(next_id)
        if next_id == tokenizer.end_id:
            break

    return generated

@torch.no_grad()
def decode_beam(model, src, beam_size=3, max_new_tokens=256):
    model.eval()
    src_dev = {k: v.to(device) for k, v in src.items()}

    beams = [([tokenizer.start_id], 0.0)]

    for _ in range(max_new_tokens):
        all_candidates = []
        for seq, score in beams:
            if seq[-1] == tokenizer.end_id:
                all_candidates.append((seq, score))
                continue

            tgt = make_empty_tgt_features(seq)
            tgt = {k: v.to(device) for k, v in tgt.items()}
            logits = model(src_dev, tgt)[0, -1]
            logp = F.log_softmax(logits, dim=-1)

            vals, idx = torch.topk(logp, k=beam_size)
            for v, i in zip(vals.tolist(), idx.tolist()):
                all_candidates.append((seq + [int(i)], score + float(v)))

        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_size]

        if all(s[-1] == tokenizer.end_id for s, _ in beams):
            break

    return beams[0][0]

## 10) Validation and Training Loop

In [ ]:
@torch.no_grad()
def evaluate_exact_match(model, loader, max_batches=None):
    model.eval()
    correct = 0
    total = 0

    for bi, batch in enumerate(loader):
        if batch is None:
            continue
        if max_batches is not None and bi >= max_batches:
            break

        src = batch['src']
        metas = batch['meta']

        # Decode one-by-one for exact grid match.
        bsz = src['token'].shape[0]
        for i in range(bsz):
            src_i = {k: v[i:i+1].to(device) for k, v in src.items()}
            pred_ids = decode_greedy(model, src_i, max_new_tokens=cfg.max_decode_tokens)
            pred_grid = tokenizer.decode_output_tokens(pred_ids)
            gt_grid = metas[i]['test_output_grid']
            correct += int(exact_grid_match(pred_grid, gt_grid))
            total += 1

    return (correct / total) if total > 0 else 0.0


def train_model(model, train_loader, val_loader, cfg):
    global optimizer, scheduler, scaler, criterion

    best_val = -1.0
    best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

    step = 0
    no_improve = 0
    losses = []

    model.train()

    for epoch in range(cfg.num_epochs):
        for batch in train_loader:
            if batch is None:
                continue

            batch = move_batch_to_device(batch, device)
            src = batch['src']
            tgt = batch['tgt']

            tgt_in, tgt_out = build_tgt_in_out(tgt)

            with autocast(enabled=(device.type == 'cuda')):
                logits = model(src, tgt_in)
                loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
                loss = loss / cfg.grad_accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % cfg.grad_accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            losses.append(float(loss.item() * cfg.grad_accum_steps))
            step += 1

            if step % 100 == 0:
                avg_loss = sum(losses[-100:]) / min(100, len(losses))
                lr = scheduler.get_last_lr()[0]
                print(f'Epoch {epoch+1} Step {step} | loss={avg_loss:.4f} | lr={lr:.6e}')

            if step % cfg.val_every_steps == 0:
                val_acc = evaluate_exact_match(model, val_loader, max_batches=50)
                print(f'Validation exact match @ step {step}: {val_acc:.4f}')

                ckpt_path = os.path.join(CHECKPOINT_DIR, f'ckpt_step_{step}.pt')
                save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, step, best_val)

                if val_acc > best_val:
                    best_val = val_acc
                    no_improve = 0
                    save_checkpoint(best_path, model, optimizer, scheduler, scaler, step, best_val)
                    print(f'New best model saved: {best_path}')
                else:
                    no_improve += 1

                model.train()

                if no_improve >= cfg.patience:
                    print('Early stopping triggered')
                    return best_path, best_val

            if step >= cfg.max_steps:
                print('Reached max training steps')
                return best_path, best_val

    return best_path, best_val

In [ ]:
best_ckpt, best_val = train_model(model, train_loader, val_loader, cfg)
print('Best checkpoint:', best_ckpt)
print('Best validation exact match:', best_val)

## 11) Optional Test-Time Training (Bonus)

In [ ]:
def build_ttt_batch_from_task(task, tokenizer, max_seq_len):
    # Build pseudo-train examples from demonstration pairs only.
    # For each demo pair, predict its output from all other demos + this demo input as test.
    samples = []
    train_pairs = task['train']
    if len(train_pairs) < 2:
        return []

    for i in range(len(train_pairs)):
        context = [train_pairs[j] for j in range(len(train_pairs)) if j != i]
        test_input = train_pairs[i]['input']
        target = train_pairs[i]['output']

        src = tokenizer.encode_source(context, test_input, max_len=max_seq_len)
        tgt = tokenizer.encode_target(target, max_len=max_seq_len)
        if src is None or tgt is None:
            continue

        samples.append({'src': src, 'tgt': tgt})

    return samples

def test_time_adapt(model, task, tokenizer, steps=10, lr=1e-5):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    samples = build_ttt_batch_from_task(task, tokenizer, cfg.max_seq_len)
    if len(samples) == 0:
        return

    for _ in range(steps):
        random.shuffle(samples)
        for s in samples:
            src = {k: torch.tensor(v, dtype=torch.long).unsqueeze(0).to(device) for k, v in s['src'].items()}
            tgt = {k: torch.tensor(v, dtype=torch.long).unsqueeze(0).to(device) for k, v in s['tgt'].items()}

            tgt_in, tgt_out = build_tgt_in_out(tgt)
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    model.eval()
print('TTT functions ready')

## 12) Final Evaluation on ARC Evaluation Set (Strict Exact Match)

In [ ]:
# Load best checkpoint before final eval
_ = load_checkpoint(best_ckpt, model, map_location=device)
model.eval()

# Build fast lookup from eval task_id -> full task for optional TTT
eval_task_lookup = {t['task_id']: t for t in eval_tasks_all}

@torch.no_grad()
def evaluate_on_eval_set(model, eval_ds):
    correct = 0
    total = 0

    # Group samples by task for optional TTT
    grouped = defaultdict(list)
    for idx, s in enumerate(eval_ds.samples):
        grouped[s['task_id']].append(idx)

    all_predictions = {}

    for task_id, sample_indices in grouped.items():
        base_state = copy.deepcopy(model.state_dict())

        if cfg.enable_ttt:
            task_obj = eval_task_lookup[task_id]
            test_time_adapt(model, task_obj, tokenizer, steps=cfg.ttt_steps, lr=cfg.ttt_lr)

        for idx in sample_indices:
            item = eval_ds[idx]
            if not item.get('valid', False):
                continue

            src = {k: torch.tensor(v, dtype=torch.long).unsqueeze(0).to(device) for k, v in item['src'].items()}

            # Attempt 1: greedy
            pred1_ids = decode_greedy(model, src, max_new_tokens=cfg.max_decode_tokens)
            pred1 = tokenizer.decode_output_tokens(pred1_ids)

            # Attempt 2: stochastic sampling (or beam fallback if invalid)
            pred2_ids = decode_sample(
                model, src,
                max_new_tokens=cfg.max_decode_tokens,
                temperature=cfg.temperature,
                top_k=cfg.top_k
            )
            pred2 = tokenizer.decode_output_tokens(pred2_ids)
            if pred2 is None:
                pred2_ids = decode_beam(model, src, beam_size=3, max_new_tokens=cfg.max_decode_tokens)
                pred2 = tokenizer.decode_output_tokens(pred2_ids)

            gt = item['test_output_grid']
            # Strict ARC rule: task case correct if any of two predictions matches exactly.
            ok = exact_grid_match(pred1, gt) or exact_grid_match(pred2, gt)

            correct += int(ok)
            total += 1

            all_predictions[(task_id, item['test_idx'])] = {
                'pred1': pred1,
                'pred2': pred2,
                'gt': gt,
                'correct': bool(ok),
            }

        # Restore original model if task-specific adaptation was used
        if cfg.enable_ttt:
            model.load_state_dict(base_state)
            model.eval()

    accuracy = (correct / total) if total > 0 else 0.0
    return accuracy, correct, total, all_predictions

eval_accuracy, eval_correct, eval_total, eval_predictions = evaluate_on_eval_set(model, eval_ds)
print('=' * 80)
print('FINAL ARC EVALUATION (EXACT MATCH, TWO PREDICTIONS)')
print(f'Correct: {eval_correct} / {eval_total}')
print(f'Accuracy: {eval_accuracy:.4f}')
print('=' * 80)

## 13) Visualization

In [ ]:
ARC_CMAP = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]

def show_grid(ax, grid, title=''):
    if grid is None:
        ax.text(0.5, 0.5, 'None', ha='center', va='center')
        ax.set_title(title)
        ax.axis('off')
        return
    arr = np.array(grid, dtype=np.int64)
    ax.imshow(arr, cmap=plt.matplotlib.colors.ListedColormap(ARC_CMAP), vmin=0, vmax=9)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

def visualize_prediction(task_id, test_idx, predictions):
    key = (task_id, test_idx)
    if key not in predictions:
        print('Prediction not found for key:', key)
        return

    p = predictions[key]

    # Recover input grid from dataset sample
    found = None
    for s in eval_ds.samples:
        if s['task_id'] == task_id and s['test_idx'] == test_idx:
            found = s
            break

    if found is None:
        print('Sample not found')
        return

    inp = found['test_input']
    gt = p['gt']
    pred = p['pred1']

    fig, axs = plt.subplots(1, 3, figsize=(10, 3.5))
    show_grid(axs[0], inp, 'Input')
    show_grid(axs[1], gt, 'Ground Truth')
    show_grid(axs[2], pred, f'Predicted (correct={p["correct"]})')
    plt.tight_layout()
    plt.show()

# Example visualization
if len(eval_predictions) > 0:
    ex_key = next(iter(eval_predictions.keys()))
    visualize_prediction(ex_key[0], ex_key[1], eval_predictions)

## 14) Ablations (Required)
Ablations included:
1. Without data augmentation
2. Without positional encoding
3. Smaller model

In [ ]:
def build_model_for_ablation(use_positional=True, small=False):
    d_model = 384 if small else cfg.d_model
    n_heads = 6 if small else cfg.num_heads
    ff = 1536 if small else cfg.feedforward_dim

    m = TransformerModel(
        vocab_size=tokenizer.vocab_size,
        d_model=d_model,
        nhead=n_heads,
        num_encoder_layers=cfg.encoder_layers,
        num_decoder_layers=cfg.decoder_layers,
        dim_feedforward=ff,
        dropout=cfg.dropout,
        max_seq_len=cfg.max_seq_len,
        max_pairs=cfg.max_pairs,
        use_positional=use_positional,
        share_encoder_layers=cfg.share_encoder_layers,
        share_decoder_layers=cfg.share_decoder_layers,
    ).to(device)
    return m

def quick_train_val(model, train_ds, val_loader, use_aug=True, steps=1500):
    loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_id),
    )

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_lambda)
    sc = GradScaler(enabled=(device.type == 'cuda'))

    model.train()
    step = 0
    while step < steps:
        for batch in loader:
            if batch is None:
                continue
            batch = move_batch_to_device(batch, device)
            tgt_in, tgt_out = build_tgt_in_out(batch['tgt'])

            with autocast(enabled=(device.type == 'cuda')):
                logits = model(batch['src'], tgt_in)
                loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))

            opt.zero_grad(set_to_none=True)
            sc.scale(loss).backward()
            sc.step(opt)
            sc.update()
            sch.step()

            step += 1
            if step >= steps:
                break

    val_acc = evaluate_exact_match(model, val_loader, max_batches=50)
    return val_acc

ablation_results = {}

if cfg.run_ablations:
    # 1) Without augmentation
    ds_no_aug = ARCDataset(
        tasks=train_tasks, tokenizer=tokenizer, max_seq_len=cfg.max_seq_len,
        training=True, use_augmentation=False, color_perm_prob=0.0
    )
    m1 = build_model_for_ablation(use_positional=True, small=False)
    ablation_results['no_augmentation'] = quick_train_val(m1, ds_no_aug, val_loader, use_aug=False, steps=1000)

    # 2) Without positional encoding
    ds_aug = ARCDataset(
        tasks=train_tasks, tokenizer=tokenizer, max_seq_len=cfg.max_seq_len,
        training=True, use_augmentation=cfg.use_augmentation, color_perm_prob=cfg.color_perm_prob
    )
    m2 = build_model_for_ablation(use_positional=False, small=False)
    ablation_results['no_positional_encoding'] = quick_train_val(m2, ds_aug, val_loader, use_aug=True, steps=1000)

    # 3) Smaller model
    m3 = build_model_for_ablation(use_positional=True, small=True)
    ablation_results['smaller_model'] = quick_train_val(m3, ds_aug, val_loader, use_aug=True, steps=1000)

print('Ablation results (validation exact match):')
for k, v in ablation_results.items():
    print(f'  {k}: {v:.4f}')

## 15) Notes on Design Choices

- The model uses the requested dimensions (512, 8 heads, 8 encoder/8 decoder, FF 2048) while sharing layer weights across depth to stay under 50M parameters.
- Sequence modeling follows ARC-style demonstrations plus test input, then autoregressive output generation.
- Two-prediction evaluation follows official exact-match logic.
- Augmentations include rotations/flips and safe color permutation (0 fixed).
- Optional TTT and beam search are included as bonus features.

This notebook is structured to run top-to-bottom in Kaggle on a T4-class GPU with mixed precision and gradient accumulation.